| Dimension    | V1 Qwen | V2 hybride | Évolution |
| ------------ | ------: | ---------: | --------: |
| Language     |    72 % |   **22 %** |  ↓ 50 pts |
| Sentiment    |    54 % |   **80 %** |  ↑ 26 pts |
| Theme        |    40 % |   **78 %** |  ↑ 38 pts |
| Product      |    58 % |   **52 %** |   ↓ 6 pts |
| Spam         |    94 % |   **88 %** |   ↓ 6 pts |
| Accord exact |     8 % |   **14 %** |   ↑ 6 pts |


Le parsing par règles résout les items explicitement décrits. Les cas restants sont principalement des informations absentes du texte (format) ou des références vagues (commande habituelle xN) pour lesquelles un LLM ne peut pas déduire de manière fiable le produit sans contexte supplémentaire.

Ce qu'on a maintenant

mart_monthly_performance donne une vue mensuelle consolidée :

| Mois  | Dépense campagne |  CA net | Jours de ventes manquants |
| ----- | ---------------: | ------: | ------------------------: |
| Jan.  |           3,33 M | 17,66 M |                         0 |
| Fév.  |           3,96 M | 16,15 M |                         0 |
| Mars  |           2,54 M | 20,80 M |                         0 |
| Avril |           2,12 M |  6,55 M |                    **14** |
| Mai   |           1,88 M | 12,49 M |                         0 |
| Juin  |           2,01 M | 14,16 M |                         0 |


Le point essentiel est avril : son niveau de ventes est directement affecté par les 14 jours sans données. Il faudra donc afficher ce manque de couverture dans le dashboard.

mart_decision_budget nous donne maintenant les indicateurs nécessaires pour la décision :

- campaign_spend_fcfa
- planned_budget_fcfa
- invoiced_budget_fcfa
- campaign_vs_invoiced_variance_fcfa
- net_revenue_fcfa
- net_units_sold
- impressions
- clicks
- blended_cpc_fcfa
- blended_cpm_fcfa
- missing_sales_days
- overall_spend_vs_plan_ratio
- revenue_to_campaign_spend_ratio

Le mart est cohérent :

- Les parts de dépenses et de budget totalisent bien 100 %.
- Meta représente la plus grosse dépense observée : 6,95 M FCFA.
- TikTok : 4,21 M FCFA.
- Radio : 2,00 M FCFA.
- Google : 1,78 M FCFA.
- Influenceurs : 0,90 M FCFA.
- Activation terrain : aucune dépense dans le campaign_spend observé.

Quelques constats descriptifs utiles :

- Meta : 6,95 M FCFA dépensés, avec des impressions/clics exploitables ; qualité moderate.
- TikTok : 4,21 M FCFA, CPM observé plus faible que Meta, mais qualité limited.
- Google : 1,78 M FCFA, CPM observé plus élevé, qualité limited.
- Radio : 2,00 M FCFA, mais pas de CPM comparable car les impressions ne sont pas disponibles.
- Influenceurs : 0,90 M FCFA, avec des métriques numériques limitées.
- Activation terrain : aucune dépense dans le campaign_spend observé, mais cela ne signifie pas nécessairement qu'aucune dépense réelle n'a eu lieu : le media_plan est justement une autre source de dépense.
Il faut maintenant éviter une erreur d'interprétation

On ne doit pas faire :

TikTok CPM < Meta CPM
        ↓
TikTok meilleur

Car les métriques disponibles et la qualité des données ne sont pas homogènes entre canaux.

Il y a plusieurs signaux importants, et surtout ils permettent de construire une allocation testable, sans prétendre faire de l'attribution causale.

1. Ce que les données montrent
Dépenses observées vs plan
| Canal              | Dépense campagne |    Plan | Part dépense | Part plan | Spend / plan |
| ------------------ | ---------------: | ------: | -----------: | --------: | -----------: |
| Meta               |          6,953 M | 6,800 M |       43,9 % |    29,3 % |      102,3 % |
| TikTok             |          4,207 M | 4,300 M |       26,6 % |    18,5 % |       97,8 % |
| Radio              |          2,000 M | 5,050 M |       12,6 % |    21,7 % |       39,6 % |
| Google             |          1,783 M | 1,780 M |       11,3 % |     7,7 % |      100,1 % |
| Influenceurs       |          0,900 M | 2,550 M |        5,7 % |    11,0 % |       35,3 % |
| Activation terrain |                0 | 2,750 M |          0 % |    11,8 % |          0 % |


Deux observations sont particulièrement importantes :

Meta représente 43,9 % des dépenses campagne contre 29,3 % dans le plan.

À l'inverse, Radio, Influenceurs et Activation terrain sont nettement sous leur plan dans l'export de campagne.

Mais attention : l'activation terrain a 2,41 M FCFA facturés dans le media plan alors que l'export campagne indique 0. Donc 0 FCFA ne doit absolument pas être interprété comme « aucun budget réel ».

C'est précisément pourquoi notre réconciliation des sources était nécessaire.

2. Les métriques média ne sont pas homogènes

On dispose de métriques numériques relativement exploitables pour :

Meta
- 5 094 567 impressions
- 68 173 clics
- CPC ≈ 102 FCFA
- CPM ≈ 1 365 FCFA
- qualité : moderate
- mesure : digital_metrics_available
TikTok
- 5 962 080 impressions
- 54 726 clics
- CPC ≈ 77 FCFA
- CPM ≈ 706 FCFA
- qualité : limited
- mesure : limited_measurement
Google
- 552 661 impressions
- 36 172 clics
- CPC ≈ 49 FCFA
- CPM ≈ 3 225 FCFA
- qualité : limited
- mesure : limited_measurement

Pour Radio, Influenceurs et Activation terrain, nous n'avons pas les mêmes métriques d'exposition/clics.

Donc on ne peut pas comparer directement les six canaux avec un même indicateur de performance.

Et surtout, un CPC plus faible ne signifie pas automatiquement davantage de ventes.

3. Le signal ventes

Le CA net observé et les dépenses campagne donnent :

| Mois    | CA / dépenses |
| ------- | ------------: |
| Janvier |          5,31 |
| Février |          4,07 |
| Mars    |      **8,15** |
| Avril   |          3,08 |
| Mai     |          6,65 |
| Juin    |          7,03 |


Mars est élevé, avril est faible.

Mais avril comporte 14 jours sans données de ventes.

Donc :

Nous ne pouvons pas utiliser avril comme preuve qu'une stratégie marketing a moins bien fonctionné.

C'est une limitation de couverture, pas un résultat marketing.

4. Customer Voice : ce que nous pouvons réellement exploiter

Sur les 2 831 commentaires :

- 1 186 positifs
- 735 négatifs
- 513 spam

Parmi les thèmes :

| Thème          | Commentaires |
| -------------- | -----------: |
| Goût (`taste`) |      **680** |
| Prix (`price`) |          376 |
| Promotion      |          313 |
| Autre          |          309 |
| Packaging      |          208 |
| Disponibilité  |          170 |
| Santé          |          152 |
| Livraison      |          109 |
| Service        |            1 |


Pour les produits explicitement identifiés :

| Produit       | Mentions |
| ------------- | -------: |
| Bouye         |  **429** |
| Bissap        |      262 |
| Gingembre     |      126 |
| Multiple      |       44 |
| Unknown       |    1 261 |
| Aucun produit |      196 |


Mais il faut être prudent : 1 261 commentaires ont un produit unknown.

Donc Customer Voice peut servir à identifier des sujets à tester, mais pas à conclure que « Bouye doit recevoir X % du budget ».

5. Ma proposition pour les 15 M FCFA

Je propose de transformer les 15 M en budget de test, plutôt qu'en budget présenté comme une allocation optimale certaine.

Allocation initiale de test  
| Canal / levier         |    Budget test | Part des 15 M |
| ---------------------- | -------------: | ------------: |
| **Meta**               |  **5 000 000** |    **33,3 %** |
| **TikTok**             |  **4 000 000** |    **26,7 %** |
| **Google**             |  **2 500 000** |    **16,7 %** |
| **Radio**              |  **1 500 000** |    **10,0 %** |
| **Influenceurs**       |  **1 000 000** |     **6,7 %** |
| **Activation terrain** |  **1 000 000** |     **6,7 %** |
| **TOTAL**              | **15 000 000** |     **100 %** |

Pourquoi cette structure ?

Meta — 5 M

Les données fournissent actuellement le niveau de mesure le plus explicite avec une qualité moderate, tout en montrant déjà une forte présence dans les dépenses historiques. Le test doit donc mesurer si cette présence produit effectivement des résultats commerciaux observables, plutôt que simplement reconduire mécaniquement sa part historique.

TikTok — 4 M

Le volume d'impressions et de clics est important et le canal est proche de son plan historique. Le CPC observé est inférieur à celui de Meta, mais cela doit rester un signal média, pas une preuve de meilleure efficacité commerciale.

Google — 2,5 M

Le canal dispose de clics mesurables et d'une dépense historique proche de son plan. Son rôle doit être mesuré séparément car son CPM observé est beaucoup plus élevé que Meta/TikTok et son contexte d'acquisition est différent.

Radio — 1,5 M

Une partie du budget est conservée pour ne pas éliminer un canal simplement parce que les métriques numériques sont absentes. Mais le test doit améliorer la mesure : dates, zones, spots, codes ou mécanisme permettant de relier l'exposition à des ventes observables.

Influenceurs — 1 M

Le budget historique campagne est inférieur au plan. Une enveloppe limitée permet de tester le levier avec des identifiants propres : influenceur, publication, période, code ou lien lorsque possible.

Activation terrain — 1 M

C'est volontairement un test limité car l'écart entre l'export campagne et le media plan montre qu'il faut d'abord réconcilier précisément ce levier. Le media plan indique d'ailleurs 2,41 M facturés malgré 0 dans l'export campagne.

6. Ce qui est important : ce n'est PAS « le classement des canaux »

La formulation dans ton livrable devra être :

« Allocation proposée pour un plan de test de 15 M FCFA »

et non :

« Les meilleurs canaux sont Meta, TikTok et Google. »

La différence est importante.

Notre donnée permet de dire :

« Ces canaux disposent actuellement de certains signaux mesurables qui permettent de construire un test. »

Elle ne permet pas de dire :

« Ces canaux ont causé le CA observé. »

7. Et surtout : le test doit être meilleur que le système actuel

C'est ici que ton projet peut devenir vraiment intéressant.

Pour les 90 prochains jours, chaque dépense doit avoir un identifiant :

campaign_id
channel
start_date
end_date
commune / zone
product
budget
impressions
clicks
reach
engagement
sales_period

Et côté ventes :

date
POS
commune
produit
unités
CA net

Puis on compare les résultats avant / pendant / après, idéalement avec des zones ou périodes comparables lorsque le design le permet.

KPI principaux

Business

- CA net
- unités vendues
- CA / jour observé
- unités / jour observé
- couverture des données

Digital

- impressions
- clics
- CPC
- CPM

Customer Voice

- sentiment
- thèmes
- produits
- volume de commentaires

Qualité

- jours de ventes manquants
- dépenses non réconciliées
- campagnes sans métriques
- commentaires non suffisamment identifiables
8. Il faut maintenant mettre cette décision dans le pipeline

Je ne conseille pas de laisser les 5 / 4 / 2,5 / 1,5 / 1 / 1 M uniquement dans le dashboard.

Mais créer une petite table décisionnelle documentée, par exemple :

decision_15m

avec :

channel
recommended_budget_fcfa
share_pct
decision_type
evidence
limitation
test_hypothesis
measurement_plan

Ainsi, le dashboard, le PDF client et le runbook pourront utiliser la même décision source.

1. Les principaux irritants identifiés

Sur les commentaires non-spam :

| Thème             | Négatifs | Positifs | Lecture factuelle                            |
| ----------------- | -------: | -------: | -------------------------------------------- |
| **Prix**          |  **347** |       27 | Forte concentration de commentaires négatifs |
| **Disponibilité** |  **151** |       10 | Problème principalement négatif              |
| **Livraison**     |       44 |       35 | Signal partagé                               |
| **Packaging**     |       61 |      145 | Majoritairement positif                      |
| **Goût**          |       14 |      578 | Très majoritairement positif                 |
| **Santé**         |       38 |       63 | Signal plutôt positif mais mixte             |
| **Promotion**     |       15 |      156 | Majoritairement positif                      |
| **Autre**         |       65 |      171 | Mixte                                        |


Deux enseignements sont particulièrement solides :

Prix : 347 négatifs sur 376 commentaires classés price, soit environ 92,3 %.

Disponibilité : 151 négatifs sur 170, soit environ 88,8 %.

À l'inverse, le goût est très majoritairement positif : 578 positifs contre seulement 14 négatifs.

2. Ce que cela change pour notre recommandation

Il ne faut surtout pas conclure :

« Le problème de prix → il faut mettre plus d'argent sur tel canal. »

Les données ne permettent pas de faire ce lien.

Elles permettent plutôt de formuler une hypothèse marketing testable :

Hypothèse H1 — Prix : une partie importante du Customer Voice exprime une sensibilité au prix. Les campagnes promotionnelles doivent donc être instrumentées pour mesurer leur impact sur les conversions/ventes.

Et :

Hypothèse H2 — Disponibilité : la disponibilité constitue également un irritant récurrent. Les activations doivent être coordonnées avec la disponibilité réelle des produits dans les points de vente.

Pendant ce temps :

Signal produit : le goût reçoit beaucoup de retours positifs. Il peut donc servir d'axe de communication à tester, mais les commentaires seuls ne prouvent pas son effet sur les ventes.

1. Lecture par produit

| Produit   | Positif | Négatif | Neutre |
| --------- | ------: | ------: | -----: |
| Bissap    |     211 |      41 |     10 |
| Bouye     |     253 |     131 |     45 |
| Gingembre |     107 |      16 |      3 |
| Multiple  |       2 |      38 |      4 |
| Unknown   |     508 |     470 |    283 |
| None      |     105 |      39 |     52 |


Points importants :

- Bissap : signal globalement positif dans les commentaires identifiés.
- Gingembre : signal également majoritairement positif.
- Bouye : beaucoup de commentaires positifs, mais aussi 131 négatifs, ce qui mérite une analyse plus poussée.
- Multiple : très peu de commentaires, donc impossible d'en tirer une conclusion solide.
- Unknown représente un volume très important : 1 261 commentaires dans la vue précédente. Il faut donc éviter de surinterpréter les comparaisons entre produits tant que l'identification produit reste limitée.
Ces chiffres décrivent la voix des clients, pas la performance commerciale des produits.

2. Ce que je mettrais dans le projet

On peut maintenant construire une chaîne beaucoup plus solide :

Signal client → hypothèse business → test mesurable

Par exemple :

Bouye

Signal : volume important de commentaires négatifs.
Hypothèse : certains aspects liés au Bouye méritent une investigation.
Test : croiser les commentaires négatifs avec les thèmes (price, availability, taste, etc.) et, si possible, avec les ventes par produit.
Décision : ne pas modifier le budget uniquement sur la base du sentiment.

Bissap / Gingembre

Signal : sentiment majoritairement positif.
Hypothèse : ces produits peuvent fournir des angles de communication intéressants.
Test : mesurer les performances des campagnes qui utilisent effectivement ces produits, sans attribuer automatiquement les ventes aux campagnes.

3. Surtout : on a maintenant une vraie histoire pour le dashboard

Le bloc Customer Voice peut présenter :

2 831 commentaires analysés

Les principaux signaux négatifs portent sur le prix et la disponibilité, tandis que les commentaires portant sur le goût sont très majoritairement positifs.
Le Bouye présente également un volume notable de commentaires négatifs, à investiguer par thème.

Puis :

Implication : utiliser ces signaux pour définir les messages et les tests marketing, mais ne pas les interpréter comme une mesure directe du ROI des campagnes.

C'est beaucoup plus défendable devant Kômian qu'un simple « le produit X est meilleur ».

Prochaine étape

Maintenant que nous avons produit × sentiment, je te propose de faire le dernier croisement utile :

produit × thème × sentiment, uniquement sur les commentaires non-spam.

Cela permettra notamment de répondre à :

- Bouye négatif : pourquoi ?
- Bissap négatif : pourquoi ?
- Gingembre négatif : pourquoi ?
- est-ce surtout le prix, la disponibilité, le goût, le packaging, etc. ?

1. Les principaux irritants identifiés

Sur les commentaires négatifs :

| Produit   | Thème         | Négatifs |
| --------- | ------------- | -------: |
| Unknown   | Prix          |      247 |
| Bouye     | Prix          |       78 |
| Unknown   | Disponibilité |       77 |
| Bissap    | Disponibilité |       34 |
| Bouye     | Packaging     |       23 |
| Gingembre | Prix          |       11 |
| Bissap    | Goût          |        5 |
| Gingembre | Goût          |        4 |


Le point important est que les problèmes ne sont pas répartis uniformément.

2. Bouye : principal signal produit identifié

Pour le Bouye, on a :

78 commentaires négatifs sur le prix
23 sur le packaging
15 sur la disponibilité
13 dans other

En parallèle, le précédent croisement montrait 253 commentaires positifs, dont 95 positifs sur le packaging.

Donc notre formulation devrait être :

Le Bouye présente un signal client contrasté : une perception positive importante du packaging, mais des irritants concentrés sur le prix et, dans une moindre mesure, la disponibilité et le packaging.

C'est plus rigoureux que de dire simplement « les clients n'aiment pas le Bouye ».

3. Bissap : problème différent

Le Bissap présente surtout :

Disponibilité négative : 34

contre seulement :

5 négatifs sur le goût

alors que le goût compte 203 commentaires positifs.

Donc on a une hypothèse intéressante :

Le frein potentiel du Bissap semble davantage lié à l'accès au produit qu'à son goût, dans les commentaires identifiés.

C'est particulièrement intéressant pour notre partie activation terrain / disponibilité POS.

4. Gingembre

Le signal est beaucoup plus limité :

- prix négatif : 11
- goût négatif : 4

Et précédemment, on avait 107 positifs sur le goût.

Donc on peut simplement retenir :

Le goût du Gingembre reçoit majoritairement des commentaires positifs dans les données identifiées.

Pas besoin d'en faire une grande conclusion.

5. Le signal transversal le plus important

Le résultat unknown / price / 247 est énorme.

Mais unknown ≠ produit.

Donc dans notre livrable, on pourra écrire :

Le prix constitue le principal irritant de la Customer Voice, mais une part importante des commentaires ne peut pas être rattachée de manière fiable à un produit.

C'est justement le genre de limite méthodologique qui montre que le projet est sérieux.

1. Ventes observées par produit

| Produit | Unités nettes |          CA net |
| ------- | ------------: | --------------: |
| BIS-1L  |        36 624 | 53 919 368 FCFA |
| GIN-1L  |        10 427 | 15 365 577 FCFA |
| BIS-33  |        20 493 | 10 089 305 FCFA |
| BOU-1L  |         3 394 |  5 688 107 FCFA |
| GIN-33  |         5 247 |  2 583 954 FCFA |

On voit donc que les données commerciales permettent maintenant de distinguer les formats/SKU, alors que la Customer Voice travaille avec des catégories produit plus générales (bissap, gingembre, bouye).

2. Croisement avec la Customer Voice

Quelques signaux ressortent :

Bissap

CA très important, notamment via BIS-1L.
Customer Voice : 203 positifs sur le goût, seulement 5 négatifs.
Principal irritant identifié : disponibilité (34 négatifs).

➡️ Hypothèse testable : pour le Bissap, l'enjeu peut être davantage de maintenir la disponibilité et convertir la demande existante que de corriger une perception gustative.

Gingembre

GIN-1L + GIN-33 représentent une activité commerciale observable.
Customer Voice : 107 positifs sur le goût, contre 4 négatifs.
11 négatifs sur le prix.

➡️ Hypothèse : tester des messages/offres liés à la valeur perçue plutôt que modifier le positionnement gustatif.

Bouye

BOU-1L : 5,69 M FCFA de CA net.
Customer Voice : 78 négatifs sur le prix, 23 sur le packaging, 15 sur la disponibilité.
Mais 253 commentaires positifs au total et 95 positifs sur le packaging.

➡️ C'est clairement un produit qui mérite une investigation ciblée, pas une conclusion simpliste du type « Bouye fonctionne mal ».

3. Une limite importante à documenter

Nous ne pouvons pas faire :

« Bouye a 78 commentaires négatifs → donc il faut réduire son budget. »

Ce serait méthodologiquement incorrect.

La Customer Voice et les ventes répondent à des questions différentes :

VENTES
→ Que s'est-il vendu ?

CUSTOMER VOICE
→ Que disent les personnes qui commentent ?

MÉDIA
→ Où et combien avons-nous dépensé ?

MESURE
→ Que se passe-t-il lorsque nous testons une action ?

Canal × produit : ce que les données montrent
| Canal  | Produit   | Dépense observée |
| ------ | --------- | ---------------: |
| TikTok | Gingembre |      1,19 M FCFA |
| Meta   | Bouye     |      1,06 M FCFA |
| Meta   | Gingembre |      1,00 M FCFA |
| TikTok | Bouye     |      0,95 M FCFA |
| Meta   | Bissap    |      0,84 M FCFA |
| TikTok | Bissap    |      0,77 M FCFA |
| Google | Bissap    |      0,28 M FCFA |

Ce qu'on peut raisonnablement en déduire

1. Meta et TikTok sont les seuls canaux où nous avons une vraie combinaison canal × produit suffisamment visible.

Ils ont également les métriques digitales que nous avons déjà observées. Cela renforce leur intérêt comme environnement de tests mesurables.

2. Bissap est présent sur les trois canaux digitaux identifiables :

- Meta
- TikTok
- Google

C'est intéressant pour construire des tests comparables, mais cela ne permet toujours pas d'attribuer les ventes de Bissap à ces canaux.

3. Bouye et Gingembre ont surtout été observés sur Meta/TikTok.

On peut donc tester des messages spécifiques par produit sur ces environnements, notamment au regard des signaux Customer Voice.

4. Et surtout : la couverture produit des campagnes est incomplète.

Le tableau ne contient que 7 combinaisons canal × produit. Pour Radio, Influenceurs et Activation terrain, aucune combinaison produit n'apparaît ici.

Donc on ne doit surtout pas écrire :

« Radio ne vend pas de Bouye »

ou :

« Activation terrain ne fonctionne pas pour Bissap ».

Les données ne permettent simplement pas de l'établir.

Nous avons maintenant assez d'éléments pour passer à la décision

Notre recommandation 15 M FCFA peut rester :

| Canal              | Allocation |
| ------------------ | ---------: |
| Meta               |  **5,0 M** |
| TikTok             |  **4,0 M** |
| Google             |  **2,5 M** |
| Radio              |  **1,5 M** |
| Influenceurs       |  **1,0 M** |
| Activation terrain |  **1,0 M** |
| **Total**          | **15,0 M** |


Mais maintenant, on peut la rendre beaucoup plus défendable :

Meta — 5 M

Pourquoi :

6,95 M FCFA de dépenses historiques observées.
Impressions/clics disponibles.
Campagnes identifiées sur les trois produits.
Permet des tests produit/message mesurables.

Condition : mesurer conversion, et pas seulement clics/impressions.

TikTok — 4 M

Pourquoi :

4,21 M FCFA de dépenses observées.
5,96 M impressions et 54 726 clics.
Campagnes identifiées sur Bissap, Bouye et Gingembre.

Condition : même logique : conversion obligatoire pour juger l'efficacité commerciale.

Google — 2,5 M

Pourquoi :

1,78 M FCFA de dépenses historiques.
36 172 clics observés.
Campagne produit identifiée pour Bissap.

Condition : suivi conversion / intention, pas seulement CPC.

Radio — 1,5 M

Allocation volontairement limitée parce que :

dépenses historiques observables ;
mais absence de métriques digitales comparables.

Condition : code, numéro, URL ou autre mécanisme de mesure.

Influenceurs — 1 M

Allocation de test :

budget prévu et dépenses observées ;
mesure actuellement limitée.

Condition : liens/codes individualisés par influenceur.

Activation terrain — 1 M

Ici, il y a une condition particulière :

Le media plan indique 2,41 M FCFA facturés, alors que l'export campaign spend indique 0 FCFA.

Donc les 1 M FCFA ne doivent être engagés qu'après réconciliation de cette divergence.

Et le Customer Voice devient notre couche d'exécution

On peut maintenant formuler les tests :

BISSAP
Goût → signal positif
Disponibilité → signal négatif
        ↓
Tester communication goût
+
vérifier disponibilité POS
BOUYE
Packaging → beaucoup de positif
Prix → principal irritant identifié
        ↓
Tester proposition de valeur / promotion
+
mesurer réaction
GINGEMBRE
Goût → signal positif
Prix → irritant secondaire
        ↓
Tester messages valeur + goût

Et surtout :

Ces signaux déterminent les hypothèses et les tests, pas directement la répartition budgétaire.